# 1. Data profiling and splitting

Este notebook crea la **fuente de datos versionada para el experimento**.

Responsabilidades:

- cargar los datos originales;
- comprobar su estructura y calidad básica;
- separar `train` y `test` de forma estratificada;
- guardar los conjuntos sin aplicar transformaciones.

> El conjunto de prueba se crea aquí, pero no se explora ni se utiliza hasta el notebook 5.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
cd "/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification"

/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification


In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [5]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"
for directory in (DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [6]:
PROJECT_DIR

PosixPath('/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification')

## Carga y contrato inicial

In [7]:
DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")
df = pd.read_csv(DATA_RAW / "breast_cancer.csv")
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [9]:
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas predictoras: {df.shape[1] - 1}")
print("Duplicados:", df.duplicated().sum())
print("Valores faltantes:", int(df.isna().sum().sum()))
print("Distribución de la clase:")
feature_names = df.drop(columns="target").columns.tolist()
target_names = ["malignant", "benign"]
display(df["target"].value_counts().rename(index=dict(enumerate(target_names))))

Filas: 569
Columnas predictoras: 30
Duplicados: 0
Valores faltantes: 0
Distribución de la clase:


,count
target,
benign,357
malignant,212


In [10]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
profile

,dtype,missing,unique,min,max
mean radius,float64,0,456,6.981000,28.11000
mean texture,float64,0,479,9.710000,39.28000
mean perimeter,float64,0,522,43.790000,188.50000
mean area,float64,0,539,143.500000,2501.00000
mean smoothness,float64,0,474,0.052630,0.16340
mean compactness,float64,0,537,0.019380,0.34540
mean concavity,float64,0,537,0.000000,0.42680
mean concave points,float64,0,542,0.000000,0.20120
mean symmetry,float64,0,432,0.106000,0.30400
mean fractal dimension,float64,0,499,0.049960,0.09744


## Split reproducible

In [11]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Proporción positiva train:", train_df["target"].mean().round(3))
print("Proporción positiva test: ", test_df["target"].mean().round(3))

Train: (455, 31)
Test: (114, 31)
Proporción positiva train: 0.626
Proporción positiva test:  0.632


In [13]:
train_df.to_csv(DATA_DIR / "train.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)

contract = {
    "dataset": "Breast Cancer Wisconsin Diagnostic",
    "target": "target",
    "target_names": target_names,
    "features": feature_names,
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "sklearn_version": sklearn.__version__,
}
(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(contract, indent=2), encoding="utf-8"
)
print("Datos y contrato guardados.")

Datos y contrato guardados.
